<a href="https://colab.research.google.com/github/anupbagale/Whole-Brain-Segmentation/blob/main/notebooks/03_structure_volumes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Structure volumes

Segmentations are often used to measure volumes, for example of the hippocampus or the ventricles.
Here the volumes from the 45-class U-Net are compared with the FreeSurfer volumes on the test subjects.
Dice and volume agreement are not the same thing: a segmentation can have a lower Dice but still get the
total volume right, or the other way round.

Run notebook 2 first. This one runs on CPU.

In [ ]:
import os, sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("wbseg"):
        !git clone -q https://github.com/anupbagale/Whole-Brain-Segmentation.git
        %cd Whole-Brain-Segmentation
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
# Change these to where the data is on your Drive
IMAGE_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Image_Folder"   # T1 images (.nii.gz)
LABEL_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Label_Folder"   # FreeSurfer aseg label maps
LUT_PATH = "/content/drive/MyDrive/FreeSurferColorLUT.txt"               # optional, only for colours
RUN_DIR = "/content/drive/MyDrive/whole_brain_seg_runs"                   # checkpoints and predictions

if not os.path.exists(LUT_PATH):
    LUT_PATH = None
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from wbseg.labels import FULL_LABELS
from wbseg.volumes import agreement, volume_table
from wbseg import plots

results = pd.read_csv("results/test_dice_full.csv")

## Agreement per structure

In [ ]:
agree = agreement(results)
agree.to_csv("results/volume_agreement.csv", index=False)
agree.round(3)

In [ ]:
fig = plots.volume_scatter(results, ["Left-Hippocampus", "Right-Hippocampus",
                                     "Left-Lateral-Ventricle", "Left-Thalamus"])
fig.savefig("figures/volume_agreement.png", dpi=150)
plt.show()

In [ ]:
fig = plots.bland_altman(results, "Left-Hippocampus")
fig.savefig("figures/bland_altman_hippocampus.png", dpi=150)
plt.show()

## Volume table for any folder of segmentations

`volume_table` reads every label map in a folder and gives one row per scan and one column per structure,
in millilitres, using the voxel size from each file's header. It works on the U-Net predictions and on
FreeSurfer's own `aseg` files.

In [ ]:
table = volume_table(f"{RUN_DIR}/full/test_predictions", FULL_LABELS)
table.to_csv("results/predicted_volumes.csv", index=False)
table.head()